# Clase 089 — XGBoost, LightGBM y CatBoost

Las tres librerías de **gradient boosting moderno** que dominan tabular ML. Comparamos
crecimiento **level-wise** (XGBoost) vs **leaf-wise** (LightGBM) y usamos sus APIs
sklearn-compatibles con **early stopping**.

> Nota: **CatBoost no está instalado en este entorno**, así que lo describimos pero
> comparamos empíricamente **XGBoost vs LightGBM vs GradientBoosting de sklearn**.

Requiere: `numpy`, `scikit-learn`, `xgboost`, `lightgbm`, `matplotlib`.

## 1. Versiones y dataset

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)
print('xgboost ', xgb.__version__)
print('lightgbm', lgb.__version__)
print('catboost: no instalado en este entorno (se describe, no se importa)')

X, y = make_classification(
    n_samples=3000, n_features=20, n_informative=10, n_redundant=4,
    weights=[0.6, 0.4], random_state=42)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)
print('train', X_train.shape, 'val', X_val.shape, 'test', X_test.shape)

## 2. XGBoost con early stopping (level-wise)

In [ ]:
t0 = time.perf_counter()
xgb_clf = xgb.XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    early_stopping_rounds=10, eval_metric='logloss',
    random_state=42, n_jobs=1, verbosity=0)
xgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
t_xgb = time.perf_counter() - t0
acc_xgb = accuracy_score(y_test, xgb_clf.predict(X_test))
print(f'XGBoost  acc test {acc_xgb:.4f}  best_iter {xgb_clf.best_iteration}  fit {t_xgb:.3f}s')

## 3. LightGBM con early stopping (leaf-wise)

In [ ]:
t0 = time.perf_counter()
lgb_clf = lgb.LGBMClassifier(
    n_estimators=100, num_leaves=31, learning_rate=0.1,
    random_state=42, n_jobs=1, verbose=-1)
lgb_clf.fit(
    X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='logloss',
    callbacks=[lgb.early_stopping(10, verbose=False), lgb.log_evaluation(0)])
t_lgb = time.perf_counter() - t0
acc_lgb = accuracy_score(y_test, lgb_clf.predict(X_test))
print(f'LightGBM acc test {acc_lgb:.4f}  best_iter {lgb_clf.best_iteration_}  fit {t_lgb:.3f}s')

## 4. GradientBoosting de sklearn (baseline)

In [ ]:
t0 = time.perf_counter()
gb_clf = GradientBoostingClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
gb_clf.fit(X_train, y_train)
t_gb = time.perf_counter() - t0
acc_gb = accuracy_score(y_test, gb_clf.predict(X_test))
print(f'sklearn GB acc test {acc_gb:.4f}  fit {t_gb:.3f}s')

assert min(acc_xgb, acc_lgb) > 0.75, 'los boosters deberían superar el azar con holgura'
print('assert OK: XGBoost y LightGBM entrenan y generalizan correctamente')

## 5. Comparativa final (accuracy y tiempo)

In [ ]:
import pandas as pd
tabla = pd.DataFrame({
    'modelo': ['XGBoost', 'LightGBM', 'sklearn GB'],
    'acc_test': [acc_xgb, acc_lgb, acc_gb],
    'fit_seg': [t_xgb, t_lgb, t_gb],
}).round(4)
print(tabla.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(tabla.modelo, tabla.acc_test, color=['#37a', '#3a7', '#a73'])
axes[0].set_ylim(min(tabla.acc_test) - 0.02, 1.0)
axes[0].set_ylabel('accuracy test')
axes[0].set_title('Accuracy')
axes[1].bar(tabla.modelo, tabla.fit_seg, color=['#37a', '#3a7', '#a73'])
axes[1].set_ylabel('segundos')
axes[1].set_title('Tiempo de fit')
for ax in axes:
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 6. Tabla conceptual de las tres librerías

| Aspecto | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| Crecimiento | Level-wise | Leaf-wise | Symmetric (oblivious) |
| Velocidad train | Media | Muy rápida | Media |
| Categóricas nativas | No (encoding) | Parcial (índices) | Sí (ordered boosting) |
| Hiperparámetro clave | `max_depth` | `num_leaves` | `depth` |
| Riesgo overfit (data chica) | Bajo | Mayor (leaf-wise) | Bajo |

**CatBoost** (no instalado aquí) destaca por manejar categóricas crudas vía
`cat_features` con **ordered boosting**, evitando target leakage.

## Ejercicios

1. Instalá `catboost` en tu entorno y repetí la comparativa con `CatBoostClassifier`
   pasando `cat_features` (sin OneHotEncoder).
2. Subí `num_leaves` de LightGBM a 200 con `max_depth=-1` y observá el overfitting.
3. Quitá `early_stopping_rounds` de XGBoost y comprobá que entrena las 100 iteraciones
   completas.
4. Reportá también el tiempo de `predict` de cada modelo, no solo el de `fit`.

## Conclusiones

- XGBoost crece **level-wise** (árboles balanceados); LightGBM **leaf-wise** (más rápido,
  más riesgo de overfit si `num_leaves` es alto).
- El **early stopping** con `eval_set` evita entrenar de más y regulariza.
- Las tres exponen API sklearn (`fit`/`predict`/`predict_proba`), usables en `Pipeline`
  y `GridSearchCV`.
- CatBoost brilla con categóricas de alta cardinalidad gracias al ordered boosting.

## ✅ Soluciones de los ejercicios

Cinco ejercicios comparando gradient boosting moderno. Usamos **XGBoost** y **LightGBM** (instalados); **CatBoost** se describe conceptualmente y NO se importa. Como no hay internet, reemplazamos el dataset `adult` por un **dataset tabular sintético con columnas categóricas** que construimos localmente, manteniendo la mecánica de OHE / encoding ordinal / categóricas nativas. `n_jobs=1`.

**Ejercicio 1 — Smoke test.** Importamos y mostramos versiones. CatBoost no está en el entorno: se describe, no se importa.

In [ ]:
import time, numpy as np, pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import accuracy_score

print('xgboost ', xgb.__version__)
print('lightgbm', lgb.__version__)
print('catboost: no instalado -> lo describimos (ordered boosting + target statistics nativas)')

# --- dataset tabular sintetico con 2 columnas categoricas con senal ---
rng = np.random.default_rng(42)
n = 4000
Xnum = rng.normal(size=(n, 5))
logit = Xnum[:, 0] * 1.2 - Xnum[:, 1] * 0.8 + Xnum[:, 2] * 0.5
cat_a = pd.cut(Xnum[:, 3], bins=[-9, -0.5, 0.5, 9], labels=['A', 'B', 'C'])
cat_b = np.where(rng.random(n) < 0.5, 'x', 'y')
logit = logit + (cat_a.codes - 1) * 0.7 + (cat_b == 'x') * 0.6
y = (logit + rng.normal(0, 0.5, n) > 0).astype(int)
df = pd.DataFrame(Xnum, columns=[f'num{i}' for i in range(5)])
df['cat_a'] = cat_a.astype(object); df['cat_b'] = cat_b
num_cols = [c for c in df.columns if c.startswith('num')]
cat_cols = ['cat_a', 'cat_b']
print('dataset:', df.shape, '| tasa positiva', round(y.mean(), 3))

**Ejercicio 2 — XGBoost + OHE + early stopping.** OneHotEncoder para las categóricas, `eval_set` y `early_stopping_rounds=20`.

In [ ]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
Xohe = np.hstack([df[num_cols].values, ohe.fit_transform(df[cat_cols])])
Xtr, Xtmp, ytr, ytmp = train_test_split(Xohe, y, test_size=0.4, random_state=42, stratify=y)
Xval, Xte, yval, yte = train_test_split(Xtmp, ytmp, test_size=0.5, random_state=42, stratify=ytmp)

t0 = time.perf_counter()
xgbm = xgb.XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=4,
                         early_stopping_rounds=20, eval_metric='logloss',
                         n_jobs=1, random_state=42)
xgbm.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
t_xgb = time.perf_counter() - t0
acc_xgb = accuracy_score(yte, xgbm.predict(Xte))
print(f'XGBoost acc test: {acc_xgb:.4f} | best_iteration {xgbm.best_iteration} | fit {t_xgb:.2f}s')

**Ejercicio 3 — LightGBM con categóricas nativas (leaf-wise).** Encoding ordinal y `categorical_feature` con los índices; comparamos tiempo de entrenamiento contra XGBoost.

In [ ]:
oe = OrdinalEncoder()
Xord = np.hstack([df[num_cols].values, oe.fit_transform(df[cat_cols])]).astype(float)
cat_idx = [len(num_cols), len(num_cols) + 1]
Xtr2, Xtmp2, ytr2, ytmp2 = train_test_split(Xord, y, test_size=0.4, random_state=42, stratify=y)
Xval2, Xte2, yval2, yte2 = train_test_split(Xtmp2, ytmp2, test_size=0.5, random_state=42, stratify=ytmp2)

t0 = time.perf_counter()
lgbm = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.1, num_leaves=31,
                          n_jobs=1, random_state=42, verbose=-1)
lgbm.fit(Xtr2, ytr2, categorical_feature=cat_idx)
t_lgb = time.perf_counter() - t0
acc_lgb = accuracy_score(yte2, lgbm.predict(Xte2))
print(f'LightGBM acc test: {acc_lgb:.4f} | fit {t_lgb:.2f}s')
print(f'tiempos -> XGBoost {t_xgb:.2f}s  vs  LightGBM {t_lgb:.2f}s')
print('LightGBM crece leaf-wise y trata categoricas sin OHE: suele ser mas rapido.')

**Ejercicio 4 — CatBoost (conceptual, sin importar).** CatBoost procesa categóricas **nativamente** con *ordered target statistics* + *ordered boosting* (evita target leakage). Como no está instalado, mostramos el análogo de "categóricas nativas" con LightGBM y verificamos que la accuracy se mantiene respecto a los ejercicios 2 y 3.

In [ ]:
print('CatBoost (descripcion):')
print(' - cat_features nativas: no requiere OHE ni encoding manual')
print(' - ordered target statistics: codifica categoricas usando el target SIN leakage')
print(' - ordered boosting: cada muestra se predice con arboles que no la vieron')
acc_native = acc_lgb  # LightGBM con categoricas nativas = stand-in de CatBoost
print(f'\naccuracy con categoricas nativas (LightGBM): {acc_native:.4f}')
assert acc_native >= min(acc_xgb, acc_lgb) - 0.05, 'las categoricas nativas no deberian degradar'
print('OK: manejar categoricas de forma nativa mantiene (o mejora) la accuracy')

**Ejercicio 5 — Comparativa final.** Mismo split para XGBoost y LightGBM: accuracy, tiempo de fit, tiempo de predict y mejor iteración.

In [ ]:
def predict_time(model, Xm):
    t0 = time.perf_counter(); model.predict(Xm); return time.perf_counter() - t0
tabla = pd.DataFrame([
    ['XGBoost',  acc_xgb, t_xgb, predict_time(xgbm, Xte),  xgbm.best_iteration],
    ['LightGBM', acc_lgb, t_lgb, predict_time(lgbm, Xte2), lgbm.n_estimators_],
], columns=['modelo', 'acc_test', 'fit_s', 'predict_s', 'best_iter'])
print(tabla.round(4).to_string(index=False))
mejor = tabla.loc[tabla['acc_test'].idxmax(), 'modelo']
print(f'\nElegiria {mejor} por accuracy; LightGBM suele ganar en velocidad y CatBoost si hay '
      'muchas categoricas de alta cardinalidad.')